# 04 · Composite detectors: best-per-category ensembles

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/jstjoe/local-privacy/blob/main/notebooks/04_composite_detectors.ipynb)

← [03_score_and_compare](03_score_and_compare.ipynb) · [series index](README.md) · [05_sanitization](05_sanitization.ipynb) →

Detectors have different strengths. A **composite** takes, for every
category, the detector that scores best on it, and merges their predictions
into one stream. No model is re-run: the composite is built from the
predictions saved in 02.

This notebook answers:

1. How much better is a composite than the best single detector?
2. Can a **local-only** composite close the gap to a hosted API?
3. How much of the gain survives on data the recipe **wasn't fitted on**?

In [ ]:
# Setup — run once per session. On Colab this clones and installs the harness
# (a few minutes the first time). Locally, after `uv sync`, it does nothing.
HARNESS_REPO = "https://github.com/jstjoe/local-privacy.git"
HARNESS_REF = "main"   # branch or tag — e.g. a PR branch to try it before merging
OPF_REPO = "https://github.com/openai/privacy-filter.git"
OPF_REF = "main"

import importlib.util, os, subprocess, sys

if importlib.util.find_spec("opf_eval") is None:
    home = "/content" if "google.colab" in sys.modules else os.path.expanduser("~")
    for repo, ref, dest in [(OPF_REPO, OPF_REF, f"{home}/privacy-filter"),
                            (HARNESS_REPO, HARNESS_REF, f"{home}/local-privacy")]:
        if not os.path.exists(dest):
            subprocess.run(["git", "clone", "-q", "--depth", "1", "--branch", ref, repo, dest], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", f"{home}/privacy-filter",
                    f"{home}/local-privacy/eval[notebooks]", f"{home}/local-privacy/api"], check=True)
    # A running kernel doesn't always see freshly pip-installed packages;
    # putting the sources on sys.path makes the imports below work either way.
    sys.path[:0] = [f"{home}/privacy-filter", f"{home}/local-privacy/eval/src", f"{home}/local-privacy/api/src"]

from opf_eval import nb
ws = nb.setup()

## Configuration

Composites are fitted and scored on disjoint halves of the sample, so use more
records than a smoke test. 200 means 100 per half, and more gives steadier
recipes.

In [ ]:
from opf_eval.detectors import registry

S = nb.load_session()

# === EDIT ME ===
N_EXAMPLES = max(S.n or 0, 200)
LOCAL_DETECTORS = ["presidio", "gliner", "gliner_nvidia", "opf"]
HOLDOUT_FRAC = 0.5          # share of records the recipe is fitted on
# ================

# Include Skyflow automatically when its credentials are available.
INCLUDE_SKYFLOW = all(nb.load_secrets(*nb.SKYFLOW_SECRETS).values())
ALL_DETECTORS = LOCAL_DETECTORS + (["skyflow"] if INCLUDE_SKYFLOW else [])
REMOTE = {d for d in ALL_DETECTORS if registry.get(d).remote}

S = nb.session(n=N_EXAMPLES, detectors=ALL_DETECTORS)
print(f"Skyflow: {'included' if INCLUDE_SKYFLOW else 'skipped (no credentials)'}")

In [ ]:
if any(d.startswith("presidio") for d in ALL_DETECTORS):
    nb.ensure_spacy_model("en_core_web_lg")
run_dir = nb.ensure_run(S)

## Holdout split

The recipe (which detector wins each category) is chosen by looking at gold
labels. If you score the composite on those same records, you get an **upper
bound**, not a prediction of how it would do on new data. So the ids are split
deterministically: the recipe is fitted on one half and scored on the other.

In [ ]:
from opf_eval import ensemble, scoring

run = scoring.Run.load(run_dir)
ALL_IDS = set(run.ids)
FIT_IDS, SCORE_IDS = scoring.holdout_split(run.ids, HOLDOUT_FRAC, seed=S.seed + 1)
print(f"fit on {len(FIT_IDS)}, score on {len(SCORE_IDS)}, overlap {len(FIT_IDS & SCORE_IDS)}")

## Baseline: individual detectors

This notebook uses the **raw** view throughout: every detector is scored on
every category the dataset annotates. A composite can cover categories that no
single detector covers, and the raw view is the view that shows it.

In [ ]:
def headline(name, ids):
    r = scoring.score_detector(run, name, view="raw", ids=ids)
    return None if r is None else {s: r.by_schema[s]["f1"] for s in ("strict", "exact", "partial", "ent_type")}

baseline = {d: headline(d, ALL_IDS) for d in ALL_DETECTORS}
nb.show_table([{"detector": d, **(f or {})} for d, f in baseline.items()],
              ["detector", "strict", "exact", "partial", "ent_type"], headers={"ent_type": "type"})

## Composite A: local detectors only

The `category_best` strategy picks each category's winner by Type F1.
Overlapping spans from different winners are resolved earliest-first, and on
ties the longer span wins.

- **upper bound:** recipe fitted on all records, scored on all records
- **holdout:** recipe fitted on the fit half, scored on the score half

In [ ]:
_, local_full, local_full_f1 = ensemble.run_category_best(
    run_dir, S.fixtures_path, excluded_detectors=REMOTE, ensemble_name="ensemble_local_full")
_, local_hold, local_hold_f1 = ensemble.run_category_best(
    run_dir, S.fixtures_path, excluded_detectors=REMOTE, ensemble_name="ensemble_local_holdout",
    fit_ids=FIT_IDS)

print("local recipe (upper bound):")
print(ensemble.format_recipe(local_full, local_full_f1))
drift = sorted(l for l in set(local_full) | set(local_hold) if local_full.get(l) != local_hold.get(l))
print(f"\nrecipe drift between full and holdout fit: {len(drift)} categories {drift}")

*Recipe drift* is the number of categories whose winner changes when the
recipe sees only half the data. High drift means the winner is decided by a
few records, and you need a bigger sample before trusting that recipe.

## Composite B: all detectors (with Skyflow)

The same two variants, with the hosted detector in the pool. If it wins almost
every category, the composite is just that detector with a few local picks
added. The categories where a local detector wins anyway are worth routing
locally.

In [ ]:
if not INCLUDE_SKYFLOW:
    print("Skyflow not included — composite B would equal composite A; skipping.")
else:
    _, all_full, all_full_f1 = ensemble.run_category_best(
        run_dir, S.fixtures_path, ensemble_name="ensemble_all_full")
    _, all_hold, _ = ensemble.run_category_best(
        run_dir, S.fixtures_path, ensemble_name="ensemble_all_holdout", fit_ids=FIT_IDS)
    print("all-detector recipe (upper bound):")
    print(ensemble.format_recipe(all_full, all_full_f1))
    won_local = sorted(l for l, d in all_full.items() if d not in REMOTE)
    print(f"\ncategories a local detector wins even with Skyflow in the pool: {won_local}")

## Headline comparison

The upper-bound rows are scored on every record. The holdout rows are scored
only on the half the recipe never saw. The "best individual local" row takes
the best local detector per column, so its winner can differ between columns.

In [ ]:
for name in ("ensemble_local_full", "ensemble_local_holdout", "ensemble_all_full", "ensemble_all_holdout"):
    if (run_dir / f"raw_{name}.jsonl").exists():
        run.reload(name)

SCHEMAS = ("strict", "exact", "partial", "ent_type")
local_scores = {d: baseline[d] for d in LOCAL_DETECTORS if baseline.get(d)}
best = {s: max(local_scores, key=lambda d: local_scores[d][s]) for s in SCHEMAS}

rows = [{"row": "best individual local", **{s: f"{local_scores[best[s]][s]:.3f} ({best[s]})" for s in SCHEMAS}}]
reference = baseline.get("skyflow") if INCLUDE_SKYFLOW else None
if reference:
    rows.append({"row": "skyflow alone", **reference})
candidates = [("composite local (upper bound)", "ensemble_local_full", ALL_IDS),
              ("composite local (holdout)", "ensemble_local_holdout", SCORE_IDS)]
if INCLUDE_SKYFLOW:
    candidates += [("composite all (upper bound)", "ensemble_all_full", ALL_IDS),
                   ("composite all (holdout)", "ensemble_all_holdout", SCORE_IDS)]
for label, name, ids in candidates:
    f1 = headline(name, ids)
    row = {"row": label, **(f1 or {})}
    if reference and f1:
        row.update({f"Δ {s}": f"{f1[s] - reference[s]:+.3f}" for s in SCHEMAS})
    rows.append(row)

cols = ["row", *SCHEMAS] + ([f"Δ {s}" for s in SCHEMAS] if reference else [])
nb.show_table(rows, cols, headers={"ent_type": "type", "Δ ent_type": "Δ type"})

## Per category

This chart shows where the composite's gains come from. A composite bar above
all of its constituents means the recipe is combining strengths.

In [ ]:
from opf_eval import plots

labels = sorted(run.dataset_labels("coarse"))
def per_label(name, ids):
    return scoring.per_label_f1(scoring.score_detector(run, name, view="raw", ids=ids), labels)

best_local = {lbl: max(per_label(d, ALL_IDS)[lbl] for d in LOCAL_DETECTORS) for lbl in labels}
series = {"best individual local": best_local}
if INCLUDE_SKYFLOW:
    series["skyflow alone"] = per_label("skyflow", ALL_IDS)
series["composite local (upper)"] = per_label("ensemble_local_full", ALL_IDS)
series["composite local (holdout)"] = per_label("ensemble_local_holdout", SCORE_IDS)
if INCLUDE_SKYFLOW:
    series["composite all (upper)"] = per_label("ensemble_all_full", ALL_IDS)
    series["composite all (holdout)"] = per_label("ensemble_all_holdout", SCORE_IDS)
plots.per_label_bars(series, labels, title=f"Composite vs individual — {run.dataset}, n={len(ALL_IDS)}");

## Reading the results

1. **Did the local composite close the gap?** Compare `composite local
   (holdout)` with the hosted detector, schema by schema.
2. **Holdout vs upper bound.** The holdout row is what you'd see in
   production. A large gap means the recipe is overfitting, so use more data.
3. **Why a composite's per-category F1 can dip below its chosen detector.**
   This is a scoring artefact, not a bug. For a given category, `nervaluate`
   counts a gold span as *missed* only if no prediction of **any** label
   overlaps it. The composite emits many labels at once, so another
   category's span can overlap a gold span the chosen detector missed. That
   turns a *miss* into an *incorrect* for this category. The per-category
   number dips, but the overall headline is the honest signal.
4. **Where to route.** Categories a local detector wins even with the hosted
   API in the pool are candidates for local handling. That saves calls, and
   may add accuracy.

## Next

Now that you can find sensitive data, decide what to replace it with.

**[05_sanitization](05_sanitization.ipynb)** — Sanitization: redact, label, number, tokenize.